<a href="https://colab.research.google.com/github/alwayzlynluv/ML-Engineering-Journey/blob/main/Data_Science/Spotify_Vibe_Clustering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Spotify Vibe Clustering




## Overview

Music streaming platforms like Spotify organize songs into playlists based on "**vibes**"—abstract moods or contexts such as chill, workout, focus, or party. In this project, I use **Unsupervised Machine Learning** to automatically cluster songs into distinct vibes based on their unique audio features.

By applying K-Means Clustering to group songs with similar mathematical profiles, I can move beyond traditional genre labels to find more nuanced musical patterns. The project also utilizes Principal Component Analysis (PCA) for **dimensionality reduction**, allowing for the visualization of high-dimensional audio data on a 2D "Vibe Map" to critically evaluate cluster quality and separation.

**Core Objectives**
- **Feature Engineering & Scaling:** Standardizing audio traits like Tempo, Loudness, and Danceability to ensure unbiased model performance.
- **Model Optimization:** Utilizing the Elbow Method and Silhouette Analysis to determine the mathematically optimal number of musical clusters.
- **Insight Discovery:** Interpreting cluster centers to assign descriptive names to each vibe, ranging from high-intensity "Power" tracks to low-energy "Ambient" soundscapes.

## Dataset

You will be able to use the **Spotify Tracks Dataset** from Kaggle, which contains audio features for thousands of tracks.

**Dataset Link:** https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset

### Key Audio Features

Each track includes the following numerical features (among others):

| Feature | Description | Range |
|---------|-------------|-------|
| `danceability` | How suitable a track is for dancing | 0.0 - 1.0 |
| `energy` | Intensity and activity level | 0.0 - 1.0 |
| `loudness` | Overall loudness in decibels | -60 - 0 dB |
| `speechiness` | Presence of spoken words | 0.0 - 1.0 |
| `acousticness` | Confidence the track is acoustic | 0.0 - 1.0 |
| `instrumentalness` | Likelihood of no vocals | 0.0 - 1.0 |
| `liveness` | Probability of live recording | 0.0 - 1.0 |
| `valence` | Musical positivity/happiness | 0.0 - 1.0 |
| `tempo` | Estimated beats per minute | ~50 - 200+ BPM |


## Part 1: Setup and Data Loading

### 1.1 Import Libraries

Import all necessary libraries for this project. You will need:
- Data manipulation: `pandas`, `numpy`
- Visualization: `matplotlib`, `seaborn`
- Machine learning: `sklearn` (specifically: `KMeans`, `PCA`, `StandardScaler`, and evaluation metrics)

**My code:**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

# Set visual style
np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_theme(style="whitegrid")

print("All required libraries for clustering loaded successfully.")

### 1.2 Load the Dataset


In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
# 1.2 Load the Dataset
df = pd.read_csv('spotify.csv')

# Display the first 5 rows to see what columns we have
print("First 5 rows of the Spotify data:")
display(df.head())

# Show how many songs and features we are working with
print(f"\nDataset Shape: {df.shape}")

# Check the data types
df.info()

# **Definitions:**

Valence: This is the "Happiness" score. High valence = happy/cheerful. Low valence = sad/angry.

Energy: Fast, loud, and noisy songs.

Acousticness: Whether the song uses electronic instruments or "real" ones.

---

## Part 2: Exploratory Data Analysis

### 2.1 Feature Selection

Select the numerical audio features that will be used for clustering. Create a new DataFrame containing only these features:
- `danceability`, `energy`, `loudness`, `speechiness`, `acousticness`, `instrumentalness`, `liveness`, `valence`, `tempo`


In [ ]:
# TODO: Create a DataFrame with only the audio features for clustering

# Defining the specific audio features mentioned
features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

# Create a new DataFrame with only these features
df_features = df[features]

# Display the first few rows of the new filtered DataFrame
print("Feature Selection Complete. New DataFrame shape:", df_features.shape)
display(df_features.head())

### 2.2 Summary Statistics

Calculate and display summary statistics for all selected features.


In [ ]:
# TODO: Display summary statistics for the audio features
#Use this syntax to see the mean, min, std, etc using the df_features from 2.1 Feature Selection
df_features.describe()


### 2.3 Distribution Visualization

Create histograms or density plots for each audio feature to understand their distributions.

In [ ]:
# TODO: Create distribution plots for each feature

# Creating a grid of histograms for all audio features
plt.figure(figsize=(15, 12))

for i, col in enumerate(df_features.columns):
    plt.subplot(3, 3, i + 1)
    sns.histplot(df_features[col], kde=True, color='skyblue')
    plt.title(f'Distribution of {col}')
    plt.tight_layout()

plt.show()

### 2.4 Correlation Analysis

Create a correlation matrix heatmap to identify relationships between features.


In [ ]:
# TODO: Create a correlation heatmap
#Heatmap is my favorite! This is a visual representation of how much two features change together

# Create a correlation matrix
corr_matrix = df_features.corr()

# Set up the matplotlib figure
plt.figure(figsize=(10, 8))

# Create a heatmap using seaborn
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)

plt.title('Correlation Heatmap of Audio Features')
plt.show()

### 2.5 EDA Discussion

**Written Response:**

Based on my exploratory analysis:

1. **Features:** It seems that tempo (ranging from 0 - 243 BPM) and loudness (ranging from -49 to +4 db) have a wide difference in scales compare to features like danceability or energy, which is between 0-1.  **Problem:** K-means uses Euclidean distance to group data. If we don't scale the data, the machine will think a difference of 50 units in tempo is 50 times more important than a difference of 1.0 in danceability. This would result in clusters based almost entirely on speed, ignoring the other "vibe" characteristics.

2. **Connection:** Based on the heatmap **energy** and **loudness** show a very strong correlation (0.76 red), and **energy** and **acousticness** (-0.73 blue) often show a strong negative correlation. **Suggestion:** This means that loud, high-intensity songs are rarely acoustic.

3. **Obversations:** Instrumentalness and speechiness is "skewed" distributions so this means they have vocals and are not podcasts/rap, creating a massive spike at the beginning of the chart. **Impact:** The spikes can be difficult for K-means to find differences in those categories unless a song is a major outliers (classical track)

---

## Part 3: Data Preprocessing

### 3.1 Handle Missing Values

Check for and handle any missing values.


In [ ]:
# TODO: Check for missing values and handle them appropriately

# Check for missing values
print(df_features.isnull().sum())

# If any are missing, drop them!!!
df_features = df_features.dropna()

### 3.2 Feature Scaling

K-means clustering uses Euclidean distance, which is sensitive to feature scales. Standardize features so each has mean 0 and standard deviation 1.

In [ ]:
# TODO: Standardize the features using StandardScaler

# Initialize the StandardScaler
scaler = StandardScaler()

# Fit and transform the feature data
# We use df_features which we cleaned in 3.1
X_scaled = scaler.fit_transform(df_features)

# Convert back to a DataFrame just to verify the math
df_scaled = pd.DataFrame(X_scaled, columns=features)

# Verify: Means should be approx 0 and Std Dev should be 1
print("Scaling Complete.")
print(f"Mean of scaled data (should be ~0): {np.mean(X_scaled):.2f}")
print(f"Std Dev of scaled data (should be 1): {np.std(X_scaled):.2f}")

# Show the first few rows to see the transformation
display(df_scaled.head())

### 3.3 Preprocessing Discussion

**Written Response:**
1. K-means relies on calculating the "Euclidean distance" (the straight-line distance) between data points to form clusters. Standardization transfom all features to the same scale, ensuring that a "vibe" is determined by all traits equally, not just the one with the biggest numbers.
2. If the data were unscaled, the tempo and loudness features would dominate the model because tempo values go up to 240 and danceability only goes to 1.0, the algorithm would treat a tiny change in tempo as m ore significant than a massive change in danceability. Fast songs vs slow songs - losing the mood.

---

## Part 4: Determining the Optimal Number of Clusters

### 4.1 The Elbow Method

Run k-means clustering for k values from 2 to 10. For each k, record the **inertia** (within-cluster sum of squares). Plot the elbow curve.


In [ ]:
# TODO: Implement the elbow method

inertia_values = []
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia_values.append(kmeans.inertia_)

# Plotting the Elbow Curve
plt.figure(figsize=(10, 6))
plt.plot(k_range, inertia_values, marker='o', linestyle='--', color='b')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal k')
plt.xticks(k_range)
plt.show()

### 4.2 Silhouette Analysis

Calculate the **silhouette score** for each value of k. The silhouette score measures how similar points are to their own cluster compared to other clusters. Plot the silhouette scores.


In [ ]:
# TODO: Calculate silhouette scores for each k

# Using a sample of 10,000 songs to speed up the calculation
from sklearn.utils import resample
X_sample = resample(X_scaled, n_samples=10000, random_state=42)

silhouette_avg_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X_sample)
    silhouette_avg = silhouette_score(X_sample, cluster_labels)
    silhouette_avg_scores.append(silhouette_avg)
    print(f"For k={k}, the average silhouette_score is: {silhouette_avg:.4f}")

# Plotting Silhouette Scores
plt.figure(figsize=(10, 6))
plt.plot(k_range, silhouette_avg_scores, marker='s', linestyle='-', color='g')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Analysis for Optimal k')
plt.xticks(k_range)
plt.show()

### 4.3 Choosing k

**Written Response :**

1. I recommend k=5 because the nn the elbow plot, where the "drop" in interia starts to flatten out. On silhoutette plot, this value provides high score to higher k-values. I think 7 would be too much.
2. The elbow chart shows gradual down because of the musicial vibes. I choose the point were adding another cluster provided "Diminishing returns" in reducing the error.
3. A silhouette score tells me the clusters have some overlap, because of the type of music but most songs are much closer to the person vibe cemter than to others.
4. Elbow methtod focuses on distance, while silhouette focuses on separation (how distinct the groups are). Fewer clusters to me means "vibes" easy to understand.

---

## Part 5: K-Means Clustering
### 5.1 Fit the Final Model

Using your chosen value of k, fit the final k-means model. Use `random_state=42` for reproducibility.

# K-Means Variation #1

In [ ]:
# TODO: Fit the final k-means model

# Initialize KMeans with your chosen k=5
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)

# Fit the model on the scaled data
kmeans.fit(X_scaled)

# Get the cluster labels
cluster_labels = kmeans.labels_

# Add the labels back to your original DataFrame (the one with song titles!)
df['cluster'] = cluster_labels

print("Final model fitted. Songs have been categorized into 5 vibes.")
print(df[['track_name', 'artists', 'cluster']].head())

# K- Means Variation #2

In [ ]:
# Variation 2: Testing k=3 to see if broader groups are better
kmeans_v2 = KMeans(n_clusters=3, random_state=42, n_init=10)

# Fit and predict
cluster_labels_v2 = kmeans_v2.fit_predict(X_scaled)

print("Variation 2 (k=3) complete.")
# This creates a temporary view to show the new labels
display(df[['track_name', 'artists']].assign(cluster_v2=cluster_labels_v2).head())

# K-Means Variation #3

In [ ]:
# Variation 3: Testing k=7 for more specific vibes
kmeans_v3 = KMeans(n_clusters=7, random_state=42, n_init=10)
cluster_labels_v3 = kmeans_v3.fit_predict(X_scaled)

print("Variation 3 (k=7) complete.")

In [ ]:
# Comparison Plot: Visualizing all 3 Variations
fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharey=True)

# Variation 1: k=5 (Your Final Choice)
sns.scatterplot(x=pca_data[:, 0], y=pca_data[:, 1], hue=cluster_labels,
                palette='viridis', ax=axes[0], s=10, alpha=0.5, legend=None)
axes[0].set_title('Variation 1: k=5 (Final Model)')
axes[0].set_xlabel('PC1 (Intensity)')
axes[0].set_ylabel('PC2 (Danceability)')

# Variation 2: k=3
sns.scatterplot(x=pca_data[:, 0], y=pca_data[:, 1], hue=cluster_labels_v2,
                palette='viridis', ax=axes[1], s=10, alpha=0.5, legend=None)
axes[1].set_title('Variation 2: k=3 (Broader Groups)')
axes[1].set_xlabel('PC1 (Intensity)')

# Variation 3: k=7
sns.scatterplot(x=pca_data[:, 0], y=pca_data[:, 1], hue=cluster_labels_v3,
                palette='viridis', ax=axes[2], s=10, alpha=0.5, legend=None)
axes[2].set_title('Variation 3: k=7 (More Specific)')
axes[2].set_xlabel('PC1 (Intensity)')

plt.suptitle('Comparison of Clustering Variations in PCA Space', fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# Summary of 3 Variations:

Center Plot (k=3): You will see large, sweeping blocks of color. It looks "cleaner," but notice how it forces very different music (like Acoustic and Classical) into the same group because it doesn't have enough categories.

Left Plot (k=5): This is my choice. The clusters are distinct but still large enough to represent a meaningful "vibe."

Right Plot (k=7): You will see the colors starting to fragment. The groups become so small that it's hard to distinguish one "vibe" from another (e.g., you might have two different "Happy" clusters that look almost identical)

### 5.2 Cluster Sizes

Examine the distribution of songs across clusters.

In [ ]:
# TODO: Analyze cluster sizes

# Count the number of songs in each cluster
cluster_counts = df['cluster'].value_counts().sort_index()

# Create a bar chart
plt.figure(figsize=(10, 6))
sns.barplot(x=cluster_counts.index, y=cluster_counts.values, palette='viridis')
plt.title('Number of Songs in Each Vibe Cluster')
plt.xlabel('Cluster ID')
plt.ylabel('Count of Songs')
plt.show()

print("Cluster Distribution:")
print(cluster_counts)

### 5.3 Cluster Centers Analysis

Examine the cluster centers to understand what characterizes each cluster.

In [ ]:
# TODO: Analyze and visualize cluster centers

# Get the centers (averages) in scaled space
centers = kmeans.cluster_centers_

# Create a heatmap to see what defines each cluster
plt.figure(figsize=(12, 6))
sns.heatmap(centers, annot=True, xticklabels=features,
            yticklabels=[f'Cluster {i}' for i in range(5)],
            cmap='RdBu_r', center=0)

plt.title('Vibe Profiles: Feature Averages by Cluster (Scaled)')
plt.show()

---

## Part 6: Dimensionality Reduction and Visualization

### 6.1 Apply PCA

Apply Principal Component Analysis (PCA) to reduce your 9 features to 2 dimensions for visualization.


In [ ]:
# TODO: Apply PCA

#Red (postive numbers) means vibe is high and blue (negative numbers) means low

# Initialize and apply PCA
pca = PCA(n_components=2)

# SCALED data (X_scaled) from part 3.2
pca_data = pca.fit_transform(X_scaled)

# Create a DataFrame for the PCA results
df_pca = pd.DataFrame(data=pca_data, columns=['PC1', 'PC2'])

print("Data reduced from 9 features to 2 principal components.")
display(df_pca.head())

### 6.2 Explained Variance

Examine how much variance is explained by the first two principal components.

In [ ]:
# TODO: Analyze explained variance
# - Print the explained variance ratio for each component
# - Print the cumulative explained variance

# Check how much information we kept
variance = pca.explained_variance_ratio_
print(f"PC1 explains {variance[0]:.2%} of the variance.")
print(f"PC2 explains {variance[1]:.2%} of the variance.")
print(f"Total variance explained: {sum(variance):.2%}")

### 6.3 Visualize Clusters in 2D

Create a scatter plot of the PCA-transformed data, colored by cluster assignment. Include cluster centers.


In [ ]:
# TODO: Create a 2D scatter plot of clusters

#PC1 means heavy music and PC2 means happiness and danceability

plt.figure(figsize=(12, 8))

# Scatter plot of the PCA results, colored by the cluster labels in Part 5
sns.scatterplot(x='PC1', y='PC2', hue=df['cluster'], data=df_pca,
                palette='viridis', alpha=0.5, s=10)

# Transform the cluster centers into the same 2D PCA space
centers_pca = pca.transform(kmeans.cluster_centers_)

# Plot the centers as large red X's
plt.scatter(centers_pca[:, 0], centers_pca[:, 1],
            marker='X', s=200, color='red', label='Vibe Centers')

plt.title('Spotify Vibe Map: 2D Projection of Music Clusters')
plt.legend(title='Vibe Cluster')
plt.show()

### 6.4 PCA Component Analysis

Examine the PCA components to understand what each principal component represents in terms of the original features.

In [ ]:
# TODO: Analyze PCA components

# Create a DataFrame of the "loadings"
loadings = pd.DataFrame(pca.components_.T, columns=['PC1', 'PC2'], index=features)

# Plotting the loadings to see which features influence each PC
plt.figure(figsize=(12, 6))
loadings.plot(kind='bar', ax=plt.gca())
plt.title('Feature Influence on Principal Components')
plt.ylabel('Weight')
plt.axhline(0, color='black', lw=1)
plt.show()

print("PC1 is most influenced by:")
print(loadings['PC1'].sort_values(ascending=False).head(3))

---

## Part 7: Cluster Interpretation and Labeling

### 7.1 Cluster Profiles

For each cluster, calculate the mean values of all features and identify the defining characteristics.


In [ ]:
# TODO: Create cluster profiles

# Sample Songs from Each Cluster
for i in range(5):
    print(f"\n--- 🎶 Sample Songs for Cluster {i} ---")
    # Pulling random samples from the original dataframe
    samples = df[df['cluster'] == i][['track_name', 'artists', 'track_genre']].sample(5)
    display(samples)

### 7.2 Sample Songs from Each Cluster

Display a few sample songs from each cluster to help validate your interpretations.

In [ ]:
# TODO: Display sample songs from each cluster
# - For each cluster, show 3-5 random tracks (with track name and artist if available)

for i in range(5):
    print(f"\n--- 🎶 Sample Songs for Cluster {i} ---")
    # Pulling random samples from the original dataframe for the current cluster
    samples = df[df['cluster'] == i][['track_name', 'artists', 'track_genre']].sample(5)
    display(samples)


### 7.3 Naming Your Vibes

**Written Response :**

Based on your analysis of cluster profiles and sample songs, assign a descriptive "vibe" name to each cluster. For each cluster, explain:



Cluster 0 (Intensity): I found Living Dead Girl and Hardcore. These are high-energy, high-loudness tracks.

Cluster 1 (Quiet): I found Continuous Rain and Ambient. These are the "zero energy" tracks.

Cluster 2 (Rhythmic): I found Trip-hop and House. These are defined by a steady beat.

Cluster 3 (Acoustic Art): I found Classical and Opera. These are high-acousticness, complex tracks.

Cluster 4 (Vocal-Heavy): I found Comedy and Rap. These are defined by high speechiness.

# 8 Limitations & Future Work

A major limitation of this project is the subjectivity of 'vibes'; audio features alone may not capture the cultural or lyrical context of a song. Additionally, K-Means assumes spherical clusters, which may not perfectly fit the overlapping nature of music genres. In the future, I would like to incorporate the Spotify API to include real-time popularity data and experiment with density-based clustering like DBSCAN to better identify outliers and non-linear music groupings.